In [4]:
from ultralytics import YOLO

# 1. Cargar el modelo preentrenado de Detección de Objetos
model = YOLO('yolov8n.pt')

# 2. Iniciar el entrenamiento usando tu archivo data.yaml
results = model.train(
    data=r'C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset1\data.yaml',
    epochs=150,
    imgsz=224,
    device=0  # Tu RTX 3060 ya está lista para trabajar
)

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lomel\Documentos\HackaTec2026\ModelosBDWaste\dataset1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

Prueba del modelo entrenado

In [ ]:
from ultralytics import YOLO

# Cargar tu modelo entrenado
mi_modelo = YOLO('runs/classify/train/weights/best.pt')

# Hacer una predicción
resultado = mi_modelo('ruta/a/imagen_de_prueba.jpg')

# Ver a qué clase pertenece
print(resultado[0].names[resultado[0].probs.top1])

In [ ]:
import cv2
import serial # Librería que usarás para comunicar tu PC con el ESP32
from ultralytics import YOLO

# 1. Cargar tu modelo entrenado
#C:\Users\lomel\runs\detect\train-2
model = YOLO(r'C:\Users\lomel\runs\detect\train-2\weights\best.pt')

# Opcional para el futuro: Conexión serial con el ESP32
# esp32 = serial.Serial('COM3', 9600) 

# 2. Iniciar la cámara web
cap = cv2.VideoCapture(0)

print("Cámara encendida. Esperando basura...")

while cap.isOpened():
    exito, frame = cap.read()
    if not exito:
        break

    # 3. Analizar el cuadro actual (verbose=False evita llenar la consola de texto basura)
    resultados = model(frame, verbose=False)

    # 4. Extraer la información de las detecciones
    for resultado in resultados:
        cajas = resultado.boxes
        
        for caja in cajas:
            # Obtener el ID de la clase y buscar su nombre real
            id_clase = int(caja.cls[0])
            nombre_basura = model.names[id_clase]
            
            # Obtener la confianza (qué tan seguro está el modelo)
            confianza = float(caja.conf[0])
            
            # Si el modelo está más del 60% seguro, tomamos acción
            if confianza > 0.60:
                print(f"¡Detectado!: {nombre_basura} (Seguridad: {confianza:.2f})")
                
                # --- AQUÍ ENTRA LA LÓGICA DE TU HARDWARE ---
                # Si tienes un ESP32 controlando un servomotor, aquí enviarías la señal:
                #
                # if nombre_basura == "organico":
                #     esp32.write(b'1')  # Señal para mover el servo a la izquierda
                # elif nombre_basura == "inorganico":
                #     esp32.write(b'2')  # Señal para mover el servo a la derecha
                # -------------------------------------------

    # 5. Mostrar la ventana con los recuadros dibujados para que veas qué pasa
    frame_anotado = resultados[0].plot()
    cv2.imshow("Ojos del Bote Inteligente", frame_anotado)

    # Presionar 'q' para apagar la cámara y cerrar el programa
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar los recursos
cap.release()
cv2.destroyAllWindows()